# 05 · Failure recovery — the resilience ladder

Transient provider errors vs confidence retries: two budgets that must never mix.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- the three-layer retry ladder, measured live
- recovery with the confidence budget untouched (the L-13 rule)
- the scars a recovered run carries
- where the wall-clock deadline and the FAILED bin fit

**Honesty label:** the flaky LLM raises a genuine `openai.APIConnectionError` so the production retry classifier treats it exactly like a provider blip.

## The ladder (as measured, not as documented)

Layer 1 — the agent belt: up to 3 attempts per node entry (`llm_retry.max_attempts` in taxonomy.yaml), exponential backoff.
Layer 2 — the graph self-loop: the node re-enters up to 2 more times (`routing._TRANSIENT_MAX_RETRIES`).
Layer 3 — the run deadline: the outer wall-clock bound (`run_limits.deadline_seconds`).
Confidence retries (`classification_attempts` / `extraction_attempts`) are SEPARATE counters — transients never spend them.

## One blip: absorbed silently by the belt

In [2]:
with lab.lab_sandbox() as env:
    flaky = lab.use_flaky_llm(env, fail_times=1)
    r = lab.run_document(env, lab.DOC_CONTRACT,
                         classification=lab.CLASSIFY_CONTRACT_HIGH,
                         extraction=lab.EXTRACT_HIGH,
                         filename="blip_once.txt")
    f = r["final"]
    print("fake LLM calls:", flaky.calls, "(1 failure + 1 retry-belt call + 1 extract)")
    print("stage:", f["stage"], "| attempts:", f["classification_attempts"])
    print("scar:", repr(f.get("error_message"))[:80])


2026-08-24 18:17:08 [warning  ] llm_retry                      agent=sorter attempt=1 backoff_s=1.13 doc_id= error=APIConnectionError matter_id=LAB-MATTER max_attempts=3 model=qwen/qwen3.7-flash run_id= trace_id= what=structured


fake LLM calls: 3 (1 failure + 1 retry-belt call + 1 extract)
stage: archived | attempts: 1
scar: None


Even a fully-absorbed blip leaves a scar: `error_message` records the
last transient error while `transient_retries_*` counters track the
self-loop usage. Ops sees the wobble in the audit trail even though the
document archived cleanly.

## Five blips: the belt AND the self-loop engage

In [3]:
with lab.lab_sandbox() as env:
    flaky = lab.use_flaky_llm(env, fail_times=5)
    r = lab.run_document(env, lab.DOC_CONTRACT,
                         classification=lab.CLASSIFY_CONTRACT_HIGH,
                         extraction=lab.EXTRACT_HIGH,
                         filename="blip_five.txt")
    f = r["final"]
    nodes = [s["node"] for s in r["steps"]]
    print("classify-document spans:", nodes.count("classify-document"),
          "(belt x3 per entry, self-looped)")
    print("classification_attempts:", f["classification_attempts"],
          "<- confidence budget UNTOUCHED")
    print("stage:", f["stage"])


2026-08-24 18:17:09 [warning  ] llm_retry                      agent=sorter attempt=1 backoff_s=0.73 doc_id= error=APIConnectionError matter_id=LAB-MATTER max_attempts=3 model=qwen/qwen3.7-flash run_id= trace_id= what=structured


2026-08-24 18:17:10 [warning  ] llm_retry                      agent=sorter attempt=2 backoff_s=1.72 doc_id= error=APIConnectionError matter_id=LAB-MATTER max_attempts=3 model=qwen/qwen3.7-flash run_id= trace_id= what=structured


2026-08-24 18:17:12 [warning  ] classify_transient_error       doc_id=3962c75a-536f-4ec6-9389-89c69ea220ac error='transient provider blip (simulated)' matter_id=LAB-MATTER run_id= trace_id= transient_retries=1


2026-08-24 18:17:12 [warning  ] transient_retry                doc_id= error='transient provider error: transient provider blip (simulated)' matter_id=LAB-MATTER retries=1 retry_target=classify run_id= trace_id=


2026-08-24 18:17:12 [warning  ] llm_retry                      agent=sorter attempt=1 backoff_s=0.72 doc_id= error=APIConnectionError matter_id=LAB-MATTER max_attempts=3 model=qwen/qwen3.7-flash run_id= trace_id= what=structured


2026-08-24 18:17:13 [warning  ] llm_retry                      agent=sorter attempt=2 backoff_s=1.46 doc_id= error=APIConnectionError matter_id=LAB-MATTER max_attempts=3 model=qwen/qwen3.7-flash run_id= trace_id= what=structured


classify-document spans: 2 (belt x3 per entry, self-looped)
classification_attempts: 1 <- confidence budget UNTOUCHED
stage: archived


THE lesson (L-13): five provider blips burned zero confidence
retries. The flaky provider got absorbed; the document still archived;
had the model ALSO been genuinely unsure, that budget was intact.

## The outer walls

Two things end a run without an archive: exhausting the transient loop sends the document to the HUMAN siding (fail-safe, not failure), and the catch-all around the whole run (deadline blowout, crash) routes to `_abort_run`: FAILED bin, tombstone manifest, `run_aborted=True`, audit row. Notebook 04 showed the other door into that bin: human rejection.

In [4]:
from graph.routing import _TRANSIENT_MAX_RETRIES
print("graph transient self-loop cap:", _TRANSIENT_MAX_RETRIES,
      "extra entries per node")


graph transient self-loop cap: 2 extra entries per node


## Where to go next

- **04 · human_in_the_loop** — the siding transient exhaustion feeds
- **06 · outputs_and_audit** — the tombstone/manifest records